### ETL in Databricks ([Dataset](https://www.kaggle.com/datasets/computingvictor/transactions-fraud-datasets))
# Gold

In [0]:
df = spark.table("silver.enriched_transactions")

## Gold Tables

### Fraud by day of week
Which day(s) of the week sees the highest number of fraudulent transactions?

In [0]:
from pyspark.sql.functions import sum

df_day = df.groupBy("day_of_week") \
    .agg(sum("is_fraud").alias("fraud_count")) \
    .orderBy("fraud_count", ascending=False)

df_day.write.mode("overwrite").saveAsTable("gold.fraud_by_day")

### Fraud rate over time (trend)

In [0]:
from pyspark.sql.functions import count,weekofyear, month, year

df_trend_month = df.groupBy(year("transaction_day").alias("year")) \
                   .agg((sum("is_fraud") / count("*")).alias("fraud_rate")) \
                   .orderBy("year")

df_trend_month.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("gold.fraud_trend")

What is the trend of the fraud rate (fraudulent transactions divided by total) over the past month?

In [0]:
from pyspark.sql.functions import col, sum, count, date_sub, max, lit
max_transaction_day = df.agg(max("transaction_day").alias("max_day")).collect()[0]["max_day"]
df_last_month = df.filter(col("transaction_day") >= date_sub(lit(max_transaction_day), 30))


df_fraud_trend = df_last_month.groupBy("transaction_day") \
                   .agg((sum("is_fraud") / count("*")).alias("fraud_rate")) \
                   .orderBy("transaction_day")

df_fraud_trend.write.mode("overwrite").saveAsTable("gold.fraud_rate_last_month")


Are there any users showing a sharp rise in transaction amount compared to their weekly average?

In [0]:
from pyspark.sql.functions import avg, when

df = df.withColumn("year", year(col("transaction_day")))
weekly_avg_df = df.groupBy("client_id", "year", "week") \
                .agg(avg("amount").alias("weekly_avg_amount"))

# Step 4: Join back to original transactions to compare each transaction to weekly average
df_enriched = df.join(weekly_avg_df, on=["client_id", "year", "week"], how="left")

# Step 5: Flag "sharp rise" transactions, e.g., 2x weekly average
df_enriched = df_enriched.withColumn(
    "sharp_rise",
    when(col("amount") >= 2 * col("weekly_avg_amount"), 1).otherwise(0)
)

sharp_rise = df_enriched.groupBy("client_id") \
                             .agg(
                                 sum("sharp_rise").alias("num_sharp_rise_transactions"),
                                 avg("amount").alias("avg_transaction_amount"),
                                 avg("weekly_avg_amount").alias("avg_weekly_amount")
                             ) \
                             .filter(col("num_sharp_rise_transactions") > 0) \
                             .orderBy(col("num_sharp_rise_transactions").desc())

# Step 7: Save as gold table
sharp_rise.write.mode("overwrite").saveAsTable("gold.sharp_rise_users")


### Top fraudulent users
Which users have the largest number of flagged (“is_fraud = true”) transactions?

In [0]:
df_users = df.filter(df.is_fraud == 1) \
    .groupBy("client_id") \
    .count() \
    .withColumnRenamed("count", "fraud_count") \
    .orderBy("fraud_count", ascending=False)

df_users.write.mode("overwrite").saveAsTable("gold.user_fraud_counts")

How many unique users commit fraudulent transactions per week?

In [0]:
from pyspark.sql.functions import countDistinct
unique_fraud_users_weekly = df.filter(col("is_fraud") == 1) \
    .groupBy("week") \
    .agg(countDistinct("client_id").alias("unique_fraud_users")) \
    .orderBy("week")

unique_fraud_users_weekly.write.mode("overwrite").saveAsTable("gold.unique_fraud_users_weekly")


### Fraud by merchant category
Which merchant categories exhibit the highest fraud rate?

Are there specific merchants with unusually high fraud volume? - MCC 5533: Automotive Parts and Accessories Stores

In [0]:
df_mcc = df.groupBy("mcc") \
    .agg(
        (sum("is_fraud") / count("*")).alias("fraud_rate")
    ) \
    .orderBy("fraud_rate", ascending=False)

df_mcc.write.mode("overwrite").saveAsTable("gold.merchant_fraud")

### Fraud by time of day
How does fraud distribution vary by time of day (morning vs night)?

In [0]:
df_time = df.groupBy("time_of_day") \
    .agg(sum("is_fraud").alias("fraud_count"))

df_time.write.mode("overwrite").saveAsTable("gold.time_of_day_fraud")

### Fraud and non-fraud amounts
What’s the average transaction amount for fraud vs non-fraud transactions?

In [0]:
fraud_avg_amount = df.groupBy("is_fraud") \
                    .agg(avg("amount").alias("avg_transaction_amount")) \
                    .orderBy("is_fraud")

fraud_avg_amount.write.mode("overwrite").saveAsTable("gold.avg_fraud_amount")


### High-value fraud
Are fraudulent transactions more common on high-value purchases compared to low-value purchases?

In [0]:
df_high = df.groupBy("is_high_value") \
    .agg(
        (sum("is_fraud") / count("*")).alias("fraud_rate")
    )

df_high.write.mode("overwrite").saveAsTable("gold.high_value_fraud")

### Total Monetary Losses Due to Fraud Each Day
What are the total monetary losses due to fraud each day?

In [0]:
fraud_loss_per_day = df.filter(col("is_fraud") == 1) \
    .groupBy("transaction_day") \
    .agg(sum("amount").alias("total_fraud_loss")) \
    .orderBy("transaction_day")

# Save as gold table
fraud_loss_per_day.write.mode("overwrite").saveAsTable("gold.fraud_loss_per_day")


### Fraud Patterns Showing Seasonal or Monthly Spikes
Do fraud patterns show seasonal or monthly spikes?

In [0]:
fraud_monthly = df.filter(col("is_fraud") == 1) \
    .groupBy("year", "month") \
    .agg(count("*").alias("num_frauds")) \
    .orderBy("year", "month")

fraud_monthly.write.mode("overwrite").saveAsTable("gold.monthly_trends")


### User Behavior Before vs After Fraudulent Event
How has user behavior changed before versus after a fraudulent event?

In [0]:
gold_user_behavior = df.groupBy("client_id", "fraud_period") \
 .agg(avg("amount").alias("avg_transaction_amount")) \
 .orderBy("client_id", "fraud_period")

gold_user_behavior.write.mode("overwrite").saveAsTable("gold.user_behavior_before_after_fraud")
